# Demographic & Economic Analysis of Polish Regions
## 01 - Data Collection

This notebook collects demographic and socioeconomic data for all 16 Polish
voivodeships from the Statistics Poland Local Data Bank (GUS BDL) API.

The analysis covers the period 2010-2025, subject to indicator availability.

### Data source

Statistics Poland - Local Data Bank (BDL) API

### Indicators

| Indicator | BDL variable ID | Period used |
|---|---:|---|
| Population | 60641 | 2010-2025 |
| Migration balance | 1365234 | 2010-2025 |
| Migration balance per 1,000 | 1365239 | 2010-2025 |
| Natural increase | 68 | 2010-2025 |
| Natural increase per 1,000 | 450551 | 2010-2025 |
| Registered unemployment rate | 461691 | 2011-2025 |
| Monthly gross salary per employee | 196229 | 2010-2024 |
| Population aged 65-69 | 72239 | 2010-2025 |
| Population aged 70+ | 72240 | 2010-2025 |

In [3]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

YEARS = range(2010, 2026)

## API helper function

The following helper function retrieves a selected BDL variable for all
voivodeships and converts the API response into a tidy pandas DataFrame.

Voivodeships are retrieved using BDL territorial level 2.


In [5]:
def fetch_bdl_variable(variable_id, column_name, years, unit_level=2):
    url = f"https://bdl.stat.gov.pl/api/v1/data/by-variable/{variable_id}"

    parameters = {
        "year": list(years),
        "unit-level": unit_level,
        "page-size": 100
    }

    response = requests.get(url, params=parameters)
    response.raise_for_status()

    data = response.json()

    rows = []

    for region in data["results"]:
        for value in region["values"]:
            rows.append({
                "unit_id": region["id"],
                "voivodeship": region["name"],
                "year": int(value["year"]),
                column_name: value["val"]
            })

    return pd.DataFrame(rows)

## Population

Variable `60641` represents the total population by place of residence,
measured as of 31 December.

In [6]:
df_population = fetch_bdl_variable(
    variable_id=60641,
    column_name="population",
    years=YEARS
)

df_population.head()

,unit_id,voivodeship,year,population
0,011200000000,MAŁOPOLSKIE,2010,3336699
1,011200000000,MAŁOPOLSKIE,2011,3346796
2,011200000000,MAŁOPOLSKIE,2012,3354077
3,011200000000,MAŁOPOLSKIE,2013,3360581
4,011200000000,MAŁOPOLSKIE,2014,3368336


In [7]:
print("Shape:", df_population.shape)
print("Voivodeships:", df_population["voivodeship"].nunique())
print("Years:", df_population["year"].min(), "-", df_population["year"].max())
print("Missing values:", df_population["population"].isna().sum())

Shape: (256, 4)
Voivodeships: 16
Years: 2010 - 2025
Missing values: 0


## Migration

Two migration indicators are collected:

- total migration balance,
- migration balance per 1000 inhabitants.

In [8]:
df_migration = fetch_bdl_variable(
    variable_id=1365234,
    column_name="migration_balance",
    years=YEARS
)

df_migration_per_1000 = fetch_bdl_variable(
    variable_id=1365239,
    column_name="migration_balance_per_1000",
    years=YEARS
)

### Migration data for 2015

Migration values for 2015 are treated as missing rather than as actual zero
values. Therefore, 2015 is excluded from analyses involving migration
indicators.

In [9]:
df_migration.loc[
    df_migration["year"] == 2015,
    "migration_balance"
] = np.nan

df_migration_per_1000.loc[
    df_migration_per_1000["year"] == 2015,
    "migration_balance_per_1000"
] = np.nan

## Natural increase

Two indicators are collected:

- total natural increase,
- natural increase per 1000 inhabitants.

In [10]:
df_natural_increase = fetch_bdl_variable(
    variable_id=68,
    column_name="natural_increase",
    years=YEARS
)

df_natural_increase_per_1000 = fetch_bdl_variable(
    variable_id=450551,
    column_name="natural_increase_per_1000",
    years=YEARS
)

## Registered unemployment rate

Variable `461691` represents the registered unemployment rate measured
in December.

The series is available from 2011, therefore unemployment is missing for 2010.

In [11]:
df_unemployment = fetch_bdl_variable(
    variable_id=461691,
    column_name="unemployment_rate",
    years=YEARS
)

## Monthly gross salary per employee

Variable `196229` represents monthly gross salary per employee within the
scope of the corresponding BDL enterprise statistics.

The available regional series covers 2010–2024. Values are nominal and are
not adjusted for inflation.

In [12]:
df_salary = fetch_bdl_variable(
    variable_id=196229,
    column_name="monthly_gross_salary_per_employee",
    years=YEARS
)

In [13]:
print("Shape:", df_salary.shape)
print("Years:", df_salary["year"].min(), "-", df_salary["year"].max())

Shape: (240, 4)
Years: 2010 - 2024


## Population aged 65+

The population aged 65+ is constructed as the sum of two BDL age groups:

- age 65–69 - variable `72239`,
- age 70+ - variable `72240`.

The share of people aged 65+ is calculated relative to the total population.

In [14]:
df_age_65_69 = fetch_bdl_variable(
    variable_id=72239,
    column_name="population_65_69",
    years=YEARS
)

df_age_70_plus = fetch_bdl_variable(
    variable_id=72240,
    column_name="population_70_plus",
    years=YEARS
)

In [15]:
df_age_65_plus = df_age_65_69.merge(
    df_age_70_plus,
    on=["unit_id", "voivodeship", "year"],
    how="inner",
    validate="one_to_one"
)

df_age_65_plus["population_65_plus"] = (
    df_age_65_plus["population_65_69"]
    + df_age_65_plus["population_70_plus"]
)

In [16]:
df_age_65_plus = df_age_65_plus.merge(
    df_population[
        ["unit_id", "voivodeship", "year", "population"]
    ],
    on=["unit_id", "voivodeship", "year"],
    how="inner",
    validate="one_to_one"
)

df_age_65_plus["population_65_plus_share"] = (
    df_age_65_plus["population_65_plus"]
    / df_age_65_plus["population"]
    * 100
)

In [17]:
df_age_65_plus = df_age_65_plus[
    [
        "unit_id",
        "voivodeship",
        "year",
        "population_65_plus",
        "population_65_plus_share"
    ]
]

## Merge datasets

All indicators are combined into a single voivodeship-year panel.

Population is used as the base dataset because it contains a complete
2010-2025 series for all 16 voivodeships.

Left joins are used for indicators with shorter time coverage so that
population observations are not removed.

In [18]:
merge_keys = ["unit_id", "voivodeship", "year"]

df_analysis = (
    df_population
    .merge(
        df_migration,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_migration_per_1000,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_natural_increase,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_natural_increase_per_1000,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_unemployment,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_salary,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_age_65_plus,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
)

## Data quality checks

The final dataset is validated before export to ensure the expected number
of observations, regions and years, as well as the absence of duplicate
voivodeship-year records.

In [19]:
print("Shape:", df_analysis.shape)
print("Voivodeships:", df_analysis["voivodeship"].nunique())
print("Years:", df_analysis["year"].min(), "-", df_analysis["year"].max())
print("Duplicates:", df_analysis.duplicated(merge_keys).sum())

display(df_analysis.isna().sum().to_frame("missing_values"))

Shape: (256, 12)
Voivodeships: 16
Years: 2010 - 2025
Duplicates: 0


,missing_values
unit_id,0
voivodeship,0
year,0
population,0
migration_balance,16
migration_balance_per_1000,16
natural_increase,0
natural_increase_per_1000,0
unemployment_rate,16
monthly_gross_salary_per_employee,16


In [20]:
assert df_analysis.shape == (256, 12)
assert df_analysis["voivodeship"].nunique() == 16
assert df_analysis["year"].nunique() == 16
assert df_analysis["year"].min() == 2010
assert df_analysis["year"].max() == 2025
assert df_analysis.duplicated(merge_keys).sum() == 0

In [21]:
assert df_analysis["unemployment_rate"].isna().sum() == 16
assert df_analysis["monthly_gross_salary_per_employee"].isna().sum() == 16
assert df_analysis["migration_balance"].isna().sum() == 16
assert df_analysis["migration_balance_per_1000"].isna().sum() == 16

## Export processed dataset

The validated dataset is saved as the analysis-ready input for the exploratory
and statistical analysis notebooks.

In [22]:
output_path = DATA_PROCESSED / "polish_regions_analysis_ready.csv"

df_analysis.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print(f"Saved: {output_path}")

Saved: ..\data\processed\polish_regions_analysis_ready.csv


In [23]:
df_final = pd.read_csv(
    output_path,
    dtype={"unit_id": str}
)

display(df_final.head())
print("Final shape:", df_final.shape)

,unit_id,voivodeship,year,population,migration_balance,migration_balance_per_1000,natural_increase,natural_increase_per_1000,unemployment_rate,monthly_gross_salary_per_employee,population_65_plus,population_65_plus_share
0,011200000000,MAŁOPOLSKIE,2010,3336699,4408.0,1.32,7416,2.23,NaN,2997.0,453027,13.577101
1,011200000000,MAŁOPOLSKIE,2011,3346796,4300.0,1.29,5797,1.74,10.5,3148.0,462310,13.813510
2,011200000000,MAŁOPOLSKIE,2012,3354077,3577.0,1.07,4529,1.35,11.4,3352.0,473819,14.126658
3,011200000000,MAŁOPOLSKIE,2013,3360581,2824.0,0.84,4030,1.20,11.5,3431.0,487081,14.493952
4,011200000000,MAŁOPOLSKIE,2014,3368336,2947.0,0.88,4808,1.43,9.7,3573.0,501278,14.882066


Final shape: (256, 12)
